In [1]:
import jax 
import jax.numpy as jnp
from core.datasetclass import TractionDataset
from core.utils import * 

/dolfinx-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
@jax.vmap
def fto3x3(f) :
    f3x3 = jnp.array([[f[0,0], f[0,1], 0.0],
                      [f[1,0], f[1,1], 0.0],
                      [0.0, 0.0, 1.0]])
    return f3x3

@jax.vmap
def transformation_jacobian(coords_elem) :
    x1, y1 = coords_elem[0]
    x2, y2 = coords_elem[1]
    x3, y3 = coords_elem[2]

    # Jacobian of shape function derivatives
    J = jnp.array([
        [x2 - x1, y2 - y1],
        [x3 - x1, y3 - y1]
    ])
    return J

@jax.vmap
def deformation_gradient_element(coords_elem, disp_elem):
    x1, y1 = coords_elem[0]
    x2, y2 = coords_elem[1]
    x3, y3 = coords_elem[2]

    # Jacobian of shape function derivatives
    J = jnp.array([
        [x2 - x1, y2 - y1],
        [x3 - x1, y3 - y1]
    ])

    # Area factor
    detJ = jnp.linalg.det(J)

    # Shape function derivatives in reference space
    dN_ref = jnp.array([
        [-1., -1.],
        [ 1.,  0.],
        [ 0.,  1.]
    ])

    # Convert to physical derivatives: dN/dx = inv(J)^T * dN_ref
    dNdx = jnp.transpose(jnp.linalg.solve(J, dN_ref.T))

    # Gradient of displacement
    gradu = disp_elem.T @ dNdx  # 2x3 @ 3x2 = 2x2

    # Deformation gradient
    F = jnp.eye(2) + gradu
    return F, dNdx

In [3]:
dataset = TractionDataset("dataset","NH")
data = dataset[1]
coords = data["mesh_pos"][:,:2]
cells = data["cells"]
u = data["u"]
node_type = data["node_type"]
load_parameter = data["load_parameter"]

coord_cells = coords[cells]
u_cells = u[cells]

F, dNdx = deformation_gradient_element(coord_cells, u_cells)

In [4]:

def MR_psi(F: jnp.ndarray, params = jnp.array([0.1, 0.1, 0.1, 0.1,0.1, 0.1,0.1, 0.1,0.1, 0.1])) -> jnp.ndarray:
    dev_params = params[:-2]
    vol_params = params[-2:]
    if F.shape[-2:] == (2, 2):
        F = jnp.array([[F[0, 0], F[0, 1], 0.], 
                    [F[1, 0], F[1, 1], 0.],
                    [0.,      0.,     1. ]])
    c = C_func(F)
    I1 = I1_func(c)
    I2 = I2_func(c)
    I3 = I3_func(c)
    I3_safe = jnp.clip(I3, 1.0e-8, 1.0e8)
    i1_dev = I3_safe**(-1/3) * I1
    i2_dev = I3_safe**(-2/3) * I2

    X = i1_dev - 3.0
    Y = i2_dev - 3.0
    
    # --- Deviatoric Terms (W) ---
    # Assuming dev_params = [c01, c02, c10, c11, c12, c20, c21, c22]
    # Using the standard N=2 Polynomial Model terms (C10, C01, C20, C11, C02)
    dev_terms = (
        # C10 * X
        dev_params[2] * X + 
        # C01 * Y
        dev_params[0] * Y + 
        # C20 * X**2
        dev_params[5] * X**2 + 
        # C11 * X * Y
        dev_params[3] * X * Y + 
        # C02 * Y**2
        dev_params[1] * Y**2 +

        dev_params[4] * X*Y**2 + 

        dev_params[6] * X**2 * Y + 

        dev_params[7] * X**2 * Y ** 2

        # Add C12, C21, C22 terms here if required by your specific model definition
    )
    
    # --- Volumetric Terms (U) ---
    # Assuming vol_params = [d0, d1] are D2 and D1 parameters (inverse bulk moduli)
    J = jnp.sqrt(I3_safe)
    J_minus_1 = J - 1.0

    # Assuming the volumetric function U(J) = (1/D1)(J-1)^2 + (1/D2)(J-1)^4
    # with D1=d1 and D2=d0 (or vice versa, depending on convention)
    
    # D1 is typically the lower order term (quadratic, hence d1)
    # D2 is typically the higher order term (quartic, hence d0)
    vol_terms = (
        # (1/D1) * (J - 1)**2
        (vol_params[0]) * J_minus_1**2
        # (1/D2) * (J - 1)**4
    )
    
    return dev_terms + vol_terms

P_mr = jax.vmap(jax.grad(MR_psi, argnums=0), in_axes=(0, None))
params = jnp.array([0.1, 0.1, 0.1, 0.1,0.1, 0.1,0.1, 0.1,0.1, 0.1])
piola = P_mr(F, params) 


In [5]:

import jax
import jax.numpy as jnp
# helper: per-element edge-based neumann traction contribution
def _neumann_cell_force(coords_el, types_el, t3, t4):
    """`
    coords_el: (3,2) float - coordinates of the 3 nodes of the element
    types_el:  (3,) int - node_type for these 3 nodes (global node_type[cells])
    t3, t4: scalars - traction magnitudes for types 3 and 4
    returns: (3,2) local nodal traction vector for this element
    """
    edges = jnp.array([[0, 1],
                       [1, 2],
                       [2, 0]])  # three local edges
    f_cell = jnp.zeros((3, 2))

    def body_fun(idx, f):
        i = edges[idx, 0]
        j = edges[idx, 1]

        ti = types_el[i]
        tj = types_el[j]

        # Only apply traction if both nodes of the edge have the same neumann type.
        is_right = (ti == 3) & (tj == 3)
        is_top   = (ti == 4) & (tj == 4)

        # choose traction vector for edge
        t_edge = jnp.where(is_right, jnp.array([t3, 0.0]),
                 jnp.where(is_top,   jnp.array([0.0, t4]),
                                         jnp.array([0.0, 0.0])))

        xi = coords_el[i]
        xj = coords_el[j]
        L = jnp.linalg.norm(xj - xi)

        # nodal contribution from this edge: each edge contributes L/2 * T to each of its two nodes
        fe_local = 0.5 * L * t_edge  # shape (2,)

        f = f.at[i].add(fe_local)
        f = f.at[j].add(fe_local)
        return f

    f_cell = jax.lax.fori_loop(0, 3, body_fun, f_cell)
    return f_cell  # (3,2)


def physical_loss(params, coords, cells, u,
                  n_nodes, node_type):
    """
    params: (mu, kappa)
    coords: (C, 3, 2) per-element nodal coords
    cells:  (C, 3) global node indices per element
    u: displacement (format compatible with deformation_gradient_element)
    reaction_forces: target reaction vector (4,) or similar used previously
    n_nodes: total number of nodes
    bc: (n_nodes, 2) boundary code mask (0 free, 1..4 etc)
    node_type: (n_nodes, 1) ints: 0 free, 1/2 fixed (dirichlet), 3 right, 4 top
    load_parameter: (2,) or (2,1) - [t3, t4]
    """

    # --- INTERNAL FORCES (unchanged) ---
    F, dNdx = deformation_gradient_element(coords, u)   # (C,2,2), (C,3,2,2?) matches your API
    dA = jnp.linalg.det(transformation_jacobian(coords)) / 2  # (C,)
    f = fto3x3(F)
    p_pos = jnp.exp(params)
    piola = P_mr(f, p_pos)[:, :2, :2]   # (C,2,2)

    # internal element nodal forces: (C,3,2)
    f_int_cell = jnp.einsum("cij, cnj -> cin", piola, dNdx) * dA[:, None, None]
    f_int_cell = jnp.swapaxes(f_int_cell, 1, 2)    # (C,3,2)

    # assemble into global internal force vector (n_nodes, 2)
    f_int_nodes = jnp.zeros((n_nodes, 2)).at[cells].add(f_int_cell)

    # --- NEUMANN EDGE-LENGTH TRACTION ---
    # normalize load_parameter to flat array
    lp = jnp.asarray(load_parameter).reshape(-1)
    t3 = 1.3
    t4 = 1.3 * 1.1
    # node_type may be (n_nodes,1) so flatten
    node_type_flat = jnp.asarray(node_type).reshape(-1)  # (n_nodes,)
    types_per_cell = node_type_flat[cells]               # (C,3)

    # vectorize per-element traction computation
    per_cell_vmap = jax.vmap(_neumann_cell_force, in_axes=(0, 0, None, None))
    f_neu_cells = per_cell_vmap(coords, types_per_cell, t3, t4)  # (C,3,2)

    # assemble global neumann nodal forces
    f_neu_nodes = jnp.zeros((n_nodes, 2)).at[cells].add(f_neu_cells)

    # --- Residual R = int(grad v : P) dx  -  int(v·T) ds(Neumann)
    R_nodes = f_int_nodes - f_neu_nodes

    # only free DOFs contribute to the residual loss (bc == 0)
    blm_loss = jnp.sum(R_nodes[(node_type != 1) & (node_type != 2)] ** 2)

    fixed_nodes_loss1 = jnp.sum((jnp.sum(R_nodes[node_type == 1], axis = 0) + jnp.sum(f_neu_nodes[node_type == 3], axis = 0))**2)
    fixed_nodes_loss2 = jnp.sum((jnp.sum(R_nodes[node_type == 2], axis = 0) + jnp.sum(f_neu_nodes[node_type == 4], axis = 0))**2)

    return blm_loss + fixed_nodes_loss1 + fixed_nodes_loss2, f_int_nodes, f_neu_nodes, R_nodes


In [10]:
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS


# -------------------------------------------
#  YOUR VFM residual (must be JAX-differentiable)
# -------------------------------------------
# example placeholder; replace with your actual VFM implementation:

def vfm_residual(theta, u_obs):
    """
    theta: (n_theta,)
    u_obs: (n_nodes * dof,)
    returns scalar R(theta) = VFM residual
    """
    u_cells = u_obs[cells]
    vfm_res = physical_loss(theta, coord_cells, cells, u_cells, coords.shape[0], node_type)[0]
    return vfm_res   # <-- replace with your VFM residual


# JIT is important for speed during HMC
vfm_residual = jax.jit(vfm_residual)


# ---------------------------------------------------------
#  NumPyro Bayesian model
# ---------------------------------------------------------

def model(u_obs):
    N_nodes = u_obs.shape[0]

    # Example prior: theta has size n_theta = 2 (change as needed)
    theta_mean = jnp.array([
        -9.17220475, -10.3874806 , -0.65999323, -10.34241358,
        -10.42296003, -10.26842128, -10.40368627, 0.39656732, 0.1
    ])

    theta_mean = jnp.array([-1,-1,-1,-1, -1,-1, -1,-1,-1])

    cov = jnp.diag(jnp.ones(9) * 0.5)   
    theta = numpyro.sample(
        "theta", 
        dist.MultivariateNormal(theta_mean, cov)
    )

    # Noise scale of VFM residual (hierarchical parameter)
    sigma_r = numpyro.sample("sigma_r", dist.HalfCauchy(1.0))

    # Compute VFM residual
    R = vfm_residual(theta, u_obs)

    # Likelihood: exp( - R / (2 sigma_r^2) )
    numpyro.factor("vfm_likelihood", -0.5 * R / (sigma_r**2))


# ---------------------------------------------------------
#  Inference
# ---------------------------------------------------------

def infer(u_obs):
    kernel = NUTS(model)
    mcmc = MCMC(kernel, num_warmup=3000, num_samples=20000)
    mcmc.run(jax.random.PRNGKey(0), u_obs=u_obs)
    return mcmc.get_samples()


# Example usage
u_obs = u
samples = infer(u_obs)

print("theta mean:", samples["theta"].mean(axis=0))
print("sigma_r mean:", samples["sigma_r"].mean())


sample: 100%|██████████| 23000/23000 [11:10<00:00, 34.29it/s, 127 steps of size 4.73e-02. acc. prob=0.82]  


theta mean: [-1.00107563 -1.01870832 -1.00001243 -1.01646185 -1.09058675 -1.02245831
 -1.36697945 -0.99266233 -1.00311991]
sigma_r mean: 1512.2489333080862


In [7]:
samples["sigma_r"].min()

Array(34.05245174, dtype=float64)

In [8]:
jnp.exp(samples["theta"])[0]

Array([0.36727772, 0.34331079, 0.37948556, 0.37231467, 0.35535372,
       0.34804746, 0.37705075, 0.37576533, 0.37619882], dtype=float64)

In [61]:
loss_ = jax.vmap(lambda params: vfm_residual(params, u_obs))

In [53]:
theta_var = jnp.var(jnp.exp(samples["theta"]), axis=0)

In [54]:
theta_var

Array([1.10969003e-11, 9.58054074e-13, 2.59575477e-04, 1.05261560e-12,
       9.03901406e-13, 1.24198901e-12, 9.42740465e-13, 2.13251834e-03,
       1.20742869e-03], dtype=float64)

In [50]:
samples["sigma_r"]

Array([3.08549016, 0.72559611, 3.26772228, ..., 3.42880499, 1.92014882,
       2.29141181], dtype=float64)

In [6]:
import jax 
import jax.numpy as jnp
import numpyro
from numpyro import distributions as dist

In [ ]:
class RBF :
    def __init__(self, scale, lengthscale) :
        self.scale = scale
        self.lengthscale = lengthscale
    def eval(self, x, x_) :
        return 

class StrainEnergyGP :
    def __init__(self) :
        self.mean_func = mu/2 * (I1_dev - 3) + kappa/2 * (jnp.sqrt(I3_dev) - 1)**2 
        self.kernel = rbf(I, I_)
    def inference(self, I_new) :


Array(1.6226422, dtype=float32)

In [29]:
import gpjax as gpx
from gpjax.mean_functions import AbstractMeanFunction

In [30]:
class NHPrior(AbstractMeanFunction) :
    def __init__(self, mu, kappa):
        super().__init__()
        self.mu = mu
        self.kappa = kappa
    def __call__(self, x):
        I1_dev = x[0]
        I3_dev = x[2]
        return self.mu/2 * (I1_dev - 3) + self.kappa/2 * (jnp.sqrt(I3_dev) - 1)**2 

In [39]:
mean_func = jax.vmap(NHPrior(1.0, 3.0))

In [40]:
kernel = gpx.kernels.RBF(active_dims=[0,1,2])  # 1-dimensional input
prior = gpx.gps.Prior(mean_function=mean_func, kernel=kernel)

In [41]:
prior(jnp.array([[3,0,1]])).sample(jax.random.PRNGKey(0))

ValueError: matmul input operand 1 must have ndim at least 1, but it has ndim 0